# FastBioDL Benchmark — Colab Edition
**Compares:** `fastbiodl` vs `sra-tools` vs `kingfisher`

**Pipeline per tool:**
| Tool | Download | Convert | Compress |
|------|----------|---------|----------|
| fastbiodl | parallel HTTP (NCBI FTP) | fasterq-dump | pigz |
| sra-tools | prefetch | fasterq-dump | pigz |
| kingfisher | kingfisher get | kingfisher convert / fasterq-dump | pigz |

Run each section top-to-bottom. The **Configuration** cell is the only one you need to edit.

## 1 · Install System Dependencies
`sra-toolkit` (prefetch, fasterq-dump), `pigz`, and build essentials.

In [ ]:
%%bash
# Update apt and install system tools
apt-get update -qq
apt-get install -y -qq \
    pigz \
    wget \
    curl \
    libncurses5 \
    build-essential

# ── sra-toolkit (prefetch + fasterq-dump) ───────────────────────────────────
SRA_VER="3.1.0"
SRA_PKG="sratoolkit.${SRA_VER}-ubuntu64"
if ! command -v prefetch &>/dev/null; then
    wget -q "https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/${SRA_VER}/${SRA_PKG}.tar.gz"
    tar -xzf "${SRA_PKG}.tar.gz"
    export PATH="${PWD}/${SRA_PKG}/bin:${PATH}"
    # Persist PATH for subsequent cells
    echo "export PATH=\"${PWD}/${SRA_PKG}/bin:\$PATH\"" >> /root/.bashrc
    echo "PATH updated: $(which prefetch)"
else
    echo "prefetch already available: $(which prefetch)"
fi

echo "=== Versions ==="
prefetch --version 2>&1 | head -2
fasterq-dump --version 2>&1 | head -2
pigz --version 2>&1

In [ ]:
import os, shutil, subprocess

# The %%bash cell above runs in a subprocess — PATH changes do NOT propagate
# back to the Python kernel.  Fix: update os.environ["PATH"] so that
# prefetch / fasterq-dump / pigz are visible to all subsequent subprocess calls.

SRA_VER = "3.1.0"
SRA_PKG = f"sratoolkit.{SRA_VER}-ubuntu64"

# Candidate locations where the bash cell may have extracted sra-toolkit
_candidates = [
    f"/content/{SRA_PKG}/bin",           # Colab default CWD
    os.path.join(os.getcwd(), SRA_PKG, "bin"),
]

for _bin in _candidates:
    if os.path.isdir(_bin) and _bin not in os.environ.get("PATH", ""):
        os.environ["PATH"] = _bin + ":" + os.environ["PATH"]
        print(f"Added to PATH: {_bin}")

# Verify the tools are now reachable
for tool in ("prefetch", "fasterq-dump", "pigz"):
    loc = shutil.which(tool)
    print(f"  {tool:20s} → {loc or 'NOT FOUND'}")
    if loc is None:
        print(f"  ⚠  {tool} not found — benchmark cells that need it will fail.")

## 2 · Install Python Dependencies

In [ ]:
%pip install -q \
    aiohttp \
    numpy \
    requests \
    scikit-learn \
    scikit-optimize \
    scipy \
    kingfisher \
    matplotlib \
    pandas

print("Python packages installed.")

## 3 · Configuration — Paths & Parameters
**Edit this cell only.** All other cells read from these variables.

In [ ]:
import os

# ════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit everything in this cell; leave the rest untouched
# ════════════════════════════════════════════════════════════════════════════

# ── Repository root (set automatically after clone in Section 4) ────────────
REPO_DIR = "/content/FastBioDL-Adaptive-Parallel_Downloader"

# ── Script paths (relative to REPO_DIR) ─────────────────────────────────────
FASTBIODL_SCRIPT    = os.path.join(REPO_DIR, "fastbiodl_upgrade.py")
SRATOOLS_SCRIPT     = os.path.join(REPO_DIR, "benchmark_sratools.py")
KINGFISHER_SCRIPT   = os.path.join(REPO_DIR, "benchmark_kingfisher.py")
COMPARE_SCRIPT      = os.path.join(REPO_DIR, "benchmark_compare.py")

# ── Accessions file ──────────────────────────────────────────────────────────
ACCESSIONS_FILE     = os.path.join(REPO_DIR, "accessions.txt")

# ── Output directories ───────────────────────────────────────────────────────
# fastbiodl
FASTBIODL_OUT_DIR   = "/content/benchmark/fastbiodl/output"

# sra-tools
SRATOOLS_SRA_DIR    = "/content/benchmark/sratools/sra"
SRATOOLS_FASTQ_DIR  = "/content/benchmark/sratools/fastq"
SRATOOLS_OUT_DIR    = "/content/benchmark/sratools/output"

# kingfisher
KINGFISHER_SRA_DIR  = "/content/benchmark/kingfisher/sra"
KINGFISHER_FASTQ_DIR = "/content/benchmark/kingfisher/fastq"
KINGFISHER_OUT_DIR  = "/content/benchmark/kingfisher/output"

# ── Performance parameters ───────────────────────────────────────────────────
THREADS             = 8          # threads for fasterq-dump, pigz, kingfisher
SEGMENT_SIZE_MB     = 128        # fastbiodl: segment size in MB
MAX_SEGMENTS        = 8          # fastbiodl: max parallel segments per file
MAX_RETRIES         = 3          # fastbiodl: retry attempts per task
KINGFISHER_METHOD   = "aws-http" # kingfisher download method: prefetch | aws-http | ena-ftp
USE_FASTQ_FTP       = False      # fastbiodl: use fastq_ftp instead of sra_ftp

# ════════════════════════════════════════════════════════════════════════════
# (nothing to edit below this line)
# ════════════════════════════════════════════════════════════════════════════

# Create all output directories now so later cells don't need to worry
_dirs = [
    FASTBIODL_OUT_DIR,
    SRATOOLS_SRA_DIR, SRATOOLS_FASTQ_DIR, SRATOOLS_OUT_DIR,
    KINGFISHER_SRA_DIR, KINGFISHER_FASTQ_DIR, KINGFISHER_OUT_DIR,
]
for _d in _dirs:
    os.makedirs(_d, exist_ok=True)

print("Configuration:")
print(f"  REPO_DIR            = {REPO_DIR}")
print(f"  FASTBIODL_SCRIPT    = {FASTBIODL_SCRIPT}")
print(f"  SRATOOLS_SCRIPT     = {SRATOOLS_SCRIPT}")
print(f"  KINGFISHER_SCRIPT   = {KINGFISHER_SCRIPT}")
print(f"  COMPARE_SCRIPT      = {COMPARE_SCRIPT}")
print(f"  ACCESSIONS_FILE     = {ACCESSIONS_FILE}")
print(f"  THREADS             = {THREADS}")
print(f"  SEGMENT_SIZE_MB     = {SEGMENT_SIZE_MB}")
print(f"  MAX_SEGMENTS        = {MAX_SEGMENTS}")
print(f"  KINGFISHER_METHOD   = {KINGFISHER_METHOD}")
print("Output directories created.")

## 4 · Upload or Clone Benchmark Scripts

In [ ]:
import subprocess, sys, os

# ── Option A (default): clone from GitHub ───────────────────────────────────
GITHUB_REPO = "https://github.com/Your-Username/FastBioDL-Adaptive-Parallel_Downloader.git"
# Replace the URL above with the actual repo URL before running.

if not os.path.isdir(REPO_DIR):
    print(f"Cloning repo into {REPO_DIR} ...")
    result = subprocess.run(
        ["git", "clone", GITHUB_REPO, REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("git clone failed — falling back to manual upload (Option B).")
        print(result.stderr[-1000:])
    else:
        print("Clone successful.")
else:
    print(f"Repo already present at {REPO_DIR}. Pulling latest changes ...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True)
    print("Done.")

# ── Option B: manual upload ──────────────────────────────────────────────────
# Uncomment the block below if you prefer to upload files directly.
# from google.colab import files
# print("Upload fastbiodl_upgrade.py, benchmark_sratools.py, "
#       "benchmark_kingfisher.py, benchmark_compare.py, "
#       "config_fastbiodl.py, utils.py, search.py, converter.py, mover.py")
# uploaded = files.upload()
# os.makedirs(REPO_DIR, exist_ok=True)
# for fname, data in uploaded.items():
#     with open(os.path.join(REPO_DIR, fname), "wb") as f:
#         f.write(data)
# print("Upload complete.")

# Verify key scripts exist
required = [FASTBIODL_SCRIPT, SRATOOLS_SCRIPT, KINGFISHER_SCRIPT, COMPARE_SCRIPT]
missing  = [p for p in required if not os.path.exists(p)]
if missing:
    print(f"WARNING: missing scripts → {missing}")
else:
    print("All required scripts found.")

## 5 · Prepare Accession List
Edit `ACCESSIONS` below, or upload your own file to `ACCESSIONS_FILE`.

In [ ]:
# ── Inline accession list ────────────────────────────────────────────────────
# Edit this list, or leave it empty to use whatever is already in ACCESSIONS_FILE.
ACCESSIONS = [
    "SRR15852393",
    "SRR15852394",
    "SRR15852396",
]

# Write to file (overwrites existing)
if ACCESSIONS:
    with open(ACCESSIONS_FILE, "w") as f:
        f.write("\n".join(ACCESSIONS) + "\n")
    print(f"Wrote {len(ACCESSIONS)} accession(s) to {ACCESSIONS_FILE}")
elif os.path.exists(ACCESSIONS_FILE):
    with open(ACCESSIONS_FILE) as f:
        ACCESSIONS = [l.strip() for l in f if l.strip()]
    print(f"Loaded {len(ACCESSIONS)} accession(s) from existing file: {ACCESSIONS_FILE}")
else:
    raise FileNotFoundError(
        f"No accessions provided and {ACCESSIONS_FILE} does not exist. "
        "Add accessions to the ACCESSIONS list above."
    )

print("Accessions:", ACCESSIONS)

# ── Optional: upload your own accessions file ────────────────────────────────
# from google.colab import files
# uploaded = files.upload()                # pick your .txt
# import shutil
# shutil.copy(list(uploaded)[0], ACCESSIONS_FILE)

## 6 · Run fastbiodl Benchmark

In [ ]:
import subprocess, sys, glob, os

# Build fastbiodl command
fastbiodl_cmd = [
    sys.executable, FASTBIODL_SCRIPT,
    "-i", ACCESSIONS_FILE,
    "-o", FASTBIODL_OUT_DIR,
    "--segment-size", str(SEGMENT_SIZE_MB),
    "--max-segments",  str(MAX_SEGMENTS),
    "--max-retries",   str(MAX_RETRIES),
]
if USE_FASTQ_FTP:
    fastbiodl_cmd.append("--fastq")

print("Running fastbiodl:")
print(" ".join(fastbiodl_cmd))
print("-" * 60)

# Stream output in real time
proc = subprocess.Popen(
    fastbiodl_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=REPO_DIR,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("-" * 60)
print(f"fastbiodl exited with code {proc.returncode}")

# Locate the results JSON produced by this run
_fb_jsons = sorted(glob.glob(os.path.join(REPO_DIR, "benchmark_fastbiodl_results_*.json")))
FASTBIODL_JSON = _fb_jsons[-1] if _fb_jsons else None
print(f"fastbiodl results JSON → {FASTBIODL_JSON}")

## 7 · Run sra-tools Benchmark

In [ ]:
import subprocess, sys, glob, os

sratools_cmd = [
    sys.executable, SRATOOLS_SCRIPT,
    "-i",            ACCESSIONS_FILE,
    "--sra-dir",     SRATOOLS_SRA_DIR,
    "--fastq-dir",   SRATOOLS_FASTQ_DIR,
    "--out-dir",     SRATOOLS_OUT_DIR,
    "--threads",     str(THREADS),
]

print("Running sra-tools benchmark:")
print(" ".join(sratools_cmd))
print("-" * 60)

proc = subprocess.Popen(
    sratools_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=REPO_DIR,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("-" * 60)
print(f"sra-tools exited with code {proc.returncode}")

_st_jsons = sorted(glob.glob(os.path.join(REPO_DIR, "benchmark_sratools_results_*.json")))
SRATOOLS_JSON = _st_jsons[-1] if _st_jsons else None
print(f"sra-tools results JSON → {SRATOOLS_JSON}")

## 8 · Run Kingfisher Benchmark

In [ ]:
import subprocess, sys, glob, os

kingfisher_cmd = [
    sys.executable, KINGFISHER_SCRIPT,
    "-i",                ACCESSIONS_FILE,
    "--sra-dir",         KINGFISHER_SRA_DIR,
    "--fastq-dir",       KINGFISHER_FASTQ_DIR,
    "--out-dir",         KINGFISHER_OUT_DIR,
    "--threads",         str(THREADS),
    "--download-method", KINGFISHER_METHOD,
]

print("Running kingfisher benchmark:")
print(" ".join(kingfisher_cmd))
print("-" * 60)

proc = subprocess.Popen(
    kingfisher_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=REPO_DIR,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("-" * 60)
print(f"kingfisher exited with code {proc.returncode}")

_kf_jsons = sorted(glob.glob(os.path.join(REPO_DIR, "benchmark_kingfisher_results_*.json")))
KINGFISHER_JSON = _kf_jsons[-1] if _kf_jsons else None
print(f"kingfisher results JSON → {KINGFISHER_JSON}")

## 9 · Load & Compare Results

In [ ]:
import json, os, subprocess, sys

benchmark_results = []

# Guard against NameError when a benchmark section was skipped
FASTBIODL_JSON  = globals().get("FASTBIODL_JSON")
SRATOOLS_JSON   = globals().get("SRATOOLS_JSON")
KINGFISHER_JSON = globals().get("KINGFISHER_JSON")

def _load_json(path, label):
    if path and os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f"[✓] Loaded {label}: {path}")
        return data
    else:
        print(f"[–] {label} result not found — skipping.")
        return None

fb_result = _load_json(FASTBIODL_JSON,  "fastbiodl")
st_result = _load_json(SRATOOLS_JSON,   "sra-tools")
kf_result = _load_json(KINGFISHER_JSON, "kingfisher")

for r in (fb_result, st_result, kf_result):
    if r:
        benchmark_results.append(r)

print(f"\nLoaded {len(benchmark_results)} result(s).")

# Run benchmark_compare.py to print the standard text table
compare_cmd = [sys.executable, COMPARE_SCRIPT]
if FASTBIODL_JSON:
    compare_cmd += ["--fastbiodl",  FASTBIODL_JSON]
if SRATOOLS_JSON:
    compare_cmd += ["--sratools",   SRATOOLS_JSON]
if KINGFISHER_JSON:
    compare_cmd += ["--kingfisher", KINGFISHER_JSON]

print("\n" + "=" * 60)
print("benchmark_compare.py output:")
print("=" * 60)
result = subprocess.run(compare_cmd, capture_output=True, text=True, cwd=REPO_DIR)
print(result.stdout)
if result.stderr:
    print(result.stderr[-2000:])

## 10 · Visualise Phase Durations (Grouped Bar Chart)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not benchmark_results:
    print("No results to plot.")
else:
    def _extract_phases(r):
        if "phases" in r:
            return r["phases"]
        t_s  = r.get("t_start", 0.0)
        t_dl = r.get("t_download_end",  t_s)
        t_fq = r.get("t_fasterq_end",   t_dl)
        t_pz = r.get("t_pigz_end",      t_fq)
        return {
            "download":    {"duration_s": round(t_dl - t_s,  2)},
            "conversion":  {"duration_s": round(t_fq - t_dl, 2)},
            "compression": {"duration_s": round(t_pz - t_fq, 2)},
        }

    tools  = [r["tool"] for r in benchmark_results]
    phases = [_extract_phases(r) for r in benchmark_results]

    dl_times = [p["download"]["duration_s"]    for p in phases]
    cv_times = [p["conversion"]["duration_s"]  for p in phases]
    cp_times = [p["compression"]["duration_s"] for p in phases]
    total    = [r["total_time_s"]              for r in benchmark_results]

    x      = np.arange(len(tools))
    width  = 0.18
    colors = ["#4C9BE8", "#F4A261", "#2A9D8F", "#E76F51"]

    fig, ax = plt.subplots(figsize=(10, 5))
    b1 = ax.bar(x - 1.5*width, dl_times, width, label="Download",    color=colors[0])
    b2 = ax.bar(x - 0.5*width, cv_times, width, label="Conversion",  color=colors[1])
    b3 = ax.bar(x + 0.5*width, cp_times, width, label="Compression", color=colors[2])
    b4 = ax.bar(x + 1.5*width, total,    width, label="Total",       color=colors[3])

    for bars in (b1, b2, b3, b4):
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2, h + 0.5,
                    f"{h:.0f}s", ha="center", va="bottom", fontsize=7
                )

    ax.set_xticks(x)
    ax.set_xticklabels(tools, fontsize=11)
    ax.set_ylabel("Wall-clock time (seconds)")
    ax.set_title("Benchmark Comparison — Phase Durations")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    _bar_path = os.path.join(REPO_DIR, "benchmark_phase_durations.png")
    plt.savefig(_bar_path, dpi=150)
    plt.show()
    print(f"Saved → {_bar_path}")

## 11 · Visualise Phase Overlap (Gantt Chart)
Shows absolute wall-clock windows. fastbiodl's phases overlap because download, conversion, and compression run concurrently. sra-tools and kingfisher run them sequentially.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if not benchmark_results:
    print("No results to plot.")
else:
    def _get_phase_windows(r):
        """Return absolute (start, end) for each phase."""
        if "phases" in r:
            ph = r["phases"]
            return {
                "download":    (ph["download"]["start"],    ph["download"]["end"]),
                "conversion":  (ph["conversion"]["start"],  ph["conversion"]["end"]),
                "compression": (ph["compression"]["start"], ph["compression"]["end"]),
            }
        # Sequential tools: reconstruct from flat timestamps
        t_s  = r.get("t_start",        0.0)
        t_dl = r.get("t_download_end",  t_s)
        t_fq = r.get("t_fasterq_end",   t_dl)
        t_pz = r.get("t_pigz_end",      t_fq)
        return {
            "download":    (t_s,  t_dl),
            "conversion":  (t_dl, t_fq),
            "compression": (t_fq, t_pz),
        }

    phase_colors = {
        "download":    "#4C9BE8",
        "conversion":  "#F4A261",
        "compression": "#2A9D8F",
    }
    phase_labels = list(phase_colors.keys())

    n_tools = len(benchmark_results)
    fig, axes = plt.subplots(n_tools, 1, figsize=(12, 2.5 * n_tools), sharex=False)
    if n_tools == 1:
        axes = [axes]

    for ax, r in zip(axes, benchmark_results):
        windows = _get_phase_windows(r)
        t_origin = windows["download"][0]   # normalise to pipeline start = 0

        row = 0
        for phase in phase_labels:
            s, e = windows[phase]
            if e > s:
                ax.broken_barh(
                    [(s - t_origin, e - s)],
                    (row - 0.4, 0.8),
                    facecolors=phase_colors[phase],
                    edgecolors="white",
                    linewidth=0.5,
                )
                mid = (s - t_origin) + (e - s) / 2
                ax.text(mid, row, f"{e-s:.0f}s", ha="center", va="center",
                        fontsize=8, color="white", fontweight="bold")
            row += 1

        ax.set_yticks(range(len(phase_labels)))
        ax.set_yticklabels([p.capitalize() for p in phase_labels])
        ax.set_xlabel("Time since pipeline start (s)")
        ax.set_title(r["tool"], fontweight="bold")
        ax.grid(axis="x", linestyle="--", alpha=0.4)

    legend_patches = [mpatches.Patch(color=c, label=p.capitalize())
                      for p, c in phase_colors.items()]
    fig.legend(handles=legend_patches, loc="upper right", framealpha=0.9)
    plt.suptitle("Phase Overlap — Gantt View", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    _gantt_path = os.path.join(REPO_DIR, "benchmark_gantt.png")
    plt.savefig(_gantt_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {_gantt_path}")

## 12 · Speedup & Per-Stage Winner Summary

In [ ]:
import pandas as pd

if not benchmark_results:
    print("No results to summarise.")
else:
    # ── Build summary DataFrame ──────────────────────────────────────────────
    def _phases_durations(r):
        if "phases" in r:
            ph = r["phases"]
            return (ph["download"]["duration_s"],
                    ph["conversion"]["duration_s"],
                    ph["compression"]["duration_s"])
        t_s  = r.get("t_start",        0.0)
        t_dl = r.get("t_download_end",  t_s)
        t_fq = r.get("t_fasterq_end",   t_dl)
        t_pz = r.get("t_pigz_end",      t_fq)
        return (round(t_dl - t_s, 2),
                round(t_fq - t_dl, 2),
                round(t_pz - t_fq, 2))

    rows = []
    for r in benchmark_results:
        dl, cv, cp = _phases_durations(r)
        rows.append({
            "Tool":           r["tool"],
            "Download (s)":   dl,
            "Conversion (s)": cv,
            "Compression (s)": cp,
            "Total (s)":      r["total_time_s"],
        })

    df = pd.DataFrame(rows).set_index("Tool")

    # ── Speedup relative to slowest run ─────────────────────────────────────
    slowest = df["Total (s)"].max()
    df["Speedup (vs slowest)"] = (slowest / df["Total (s)"]).round(2)

    # ── Per-stage winner ─────────────────────────────────────────────────────
    stage_cols = ["Download (s)", "Conversion (s)", "Compression (s)", "Total (s)"]
    winners = {col: df[col].idxmin() for col in stage_cols}

    print("=" * 65)
    print(" BENCHMARK SPEEDUP SUMMARY")
    print("=" * 65)
    display(df.style
              .highlight_min(subset=stage_cols, color="#d4edda", axis=0)
              .format("{:.1f}", subset=stage_cols)
              .format("{:.2f}x", subset=["Speedup (vs slowest)"])
              .set_caption("Green = fastest for that stage"))

    print("\nPer-stage winner:")
    for col, winner in winners.items():
        val = df.loc[winner, col]
        print(f"  {col:<20} → {winner}  ({val:.1f}s)")